# Limpieza de nacionalidad..

## 1. Carga de librerías

In [1]:
import pandas as pd
import numpy as np

## 2. Carga del dataset 

In [2]:
df_nac_raw = pd.read_csv("../data_raw/nacionalidad.csv", sep=";", low_memory=False)

## 3. Exploración inicial

In [3]:
df_nac_raw.head()

,Sexo,Municipios,Nacionalidad,Edad (grandes grupos),Periodo,Total
0,Total,Total Nacional,Total,Todas las edades,1 de enero de 2022,47.475.420
1,Total,Total Nacional,Total,Todas las edades,1 de enero de 2021,47.385.107
2,Total,Total Nacional,Total,Todas las edades,1 de enero de 2020,47.450.795
3,Total,Total Nacional,Total,Todas las edades,1 de enero de 2019,47.026.208
4,Total,Total Nacional,Total,Todas las edades,1 de enero de 2018,46.722.980


In [4]:
df_nac_raw.shape

(5857920, 6)

In [5]:
df_nac_raw.dtypes

Sexo                     object
Municipios               object
Nacionalidad             object
Edad (grandes grupos)    object
Periodo                  object
Total                    object
dtype: object

## 4. Normalizar nombres de columnas 

In [6]:
df_nac_raw.columns = (
    df_nac_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

## 5. Ver variables clave

In [7]:
df_nac_raw['sexo'].value_counts()

sexo
Total      1952640
Hombres    1952640
Mujeres    1952640
Name: count, dtype: int64

In [8]:
df_nac_raw['nacionalidad'].unique()[:10]

array(['Total', 'Española', 'Extranjera'], dtype=object)

In [9]:
df_nac_raw['periodo'].unique()[:20]

array(['1 de enero de 2022', '1 de enero de 2021', '1 de enero de 2020',
       '1 de enero de 2019', '1 de enero de 2018', '1 de enero de 2017',
       '1 de enero de 2016', '1 de enero de 2015', '1 de enero de 2014',
       '1 de enero de 2013', '1 de enero de 2012', '1 de enero de 2011',
       '1 de enero de 2010', '1 de enero de 2009', '1 de enero de 2008',
       '1 de enero de 2007', '1 de enero de 2006', '1 de enero de 2005',
       '1 de enero de 2004', '1 de enero de 2003'], dtype=object)

## 6. Filtrado de población total

In [10]:
# No necesitamos separar por sexo, así que nos quedamos solo con 'Total
df_nac = df_nac_raw[df_nac_raw['sexo'] == "Total"].copy()

In [11]:
df_nac.shape

(1952640, 6)

## 7. Eliminacion de "Total" en nacionalidad

In [12]:
# Eliminamos la categoría "Total"
df_nac = df_nac[df_nac['nacionalidad'] != "Total"]

In [13]:
df_nac['nacionalidad'].unique()

array(['Española', 'Extranjera'], dtype=object)

## 8. Limpieza de la columna 'periodo'

In [14]:
# Extraemos solo el año
df_nac['periodo'] = df_nac['periodo'].str[-4:].astype(int)

In [15]:
df_nac['periodo'].unique()[:21]

array([2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012,
       2011, 2010, 2009, 2008, 2007, 2006, 2005, 2004, 2003])

## 9. Conversión de población

In [16]:
# Hacemos lo mismo que ne los otros notebooks para "poblacion"
df_nac['total'] = (
    df_nac['total']
    .astype(str)
    .str.replace(".", "", regex=False)
)

In [17]:
df_nac['total'] = pd.to_numeric(df_nac['total'], errors="coerce")

In [18]:
df_nac['total'].isna().sum()

np.int64(2664)

## 10. Tratamiento de valores faltantes

### Valores faltantes en población

Al revisar la columna `total` se detectaron algunos valores nulos (NaN).
Estos casos corresponden a municipios y años en los que no hay dato de población
disponible en el dataset original.

Dado que no es posible analizar la evolución de la población sin ese valor,
se decidió eliminar estas observaciones del conjunto de datos.

In [19]:
# Eliminamos filas sin dato de población
df_nac = df_nac.dropna(subset=['total'])

In [20]:
df_nac['total'].isna().sum()

np.int64(0)

## 11. Eliminación de agregados territoriales

In [21]:
# Eliminamos agregados territoriales como "Total Nacional"
df_nac = df_nac[df_nac['municipios'] != "Total Nacional"]

In [22]:
df_nac['municipios'].head()

320    44001 Ababuj
321    44001 Ababuj
322    44001 Ababuj
323    44001 Ababuj
324    44001 Ababuj
Name: municipios, dtype: object

## 12. Separación de código y nombre del municipio

In [23]:
# Separamos el código INE del nombre del municipio
df_nac[['codigo_municipio', 'municipio']] = df_nac['municipios'].str.split(" ", n=1, expand=True)

In [24]:
# Reordenamos columnas
cols = ["codigo_municipio", "municipio"] + [col for col in df_nac.columns if col not in ["codigo_municipio", "municipio"]]
df_nac = df_nac[cols]

In [25]:
df_nac.head()

,codigo_municipio,municipio,sexo,municipios,nacionalidad,edad_(grandes_grupos),periodo,total
320,44001,Ababuj,Total,44001 Ababuj,Española,Todas las edades,2022,71.0
321,44001,Ababuj,Total,44001 Ababuj,Española,Todas las edades,2021,75.0
322,44001,Ababuj,Total,44001 Ababuj,Española,Todas las edades,2020,76.0
323,44001,Ababuj,Total,44001 Ababuj,Española,Todas las edades,2019,72.0
324,44001,Ababuj,Total,44001 Ababuj,Española,Todas las edades,2018,73.0


## 13. Filtrado por edad

In [26]:
# Filtramos por "Todas las edades" para identificar qué filas conservar antes de borrarla
df_nac = df_nac[df_nac['edad_(grandes_grupos)'] == "Todas las edades"]

## 14. Limpieza final de columnas

In [27]:
# Eliminamos columnas que ya no aportan información
df_nac = df_nac.drop(columns=['sexo', 'municipios', 'edad_(grandes_grupos)'])

In [28]:
df_nac.columns, df_nac.head()

(Index(['codigo_municipio', 'municipio', 'nacionalidad', 'periodo', 'total'], dtype='object'),
     codigo_municipio municipio nacionalidad  periodo  total
 320            44001    Ababuj     Española     2022   71.0
 321            44001    Ababuj     Española     2021   75.0
 322            44001    Ababuj     Española     2020   76.0
 323            44001    Ababuj     Española     2019   72.0
 324            44001    Ababuj     Española     2018   73.0)

In [29]:
df_nac['codigo_municipio'] = df_nac['codigo_municipio'].astype(str).str.zfill(5)

## 15. Renombrar columna de población

In [30]:
# Renombramos la columna para mantener consistencia
df_nac = df_nac.rename(columns={'total': 'poblacion'})

## 16. Reordenar columnas

In [31]:
# Queremos ponerlo todo en el mismo orden en todos los datasets
df_nac = df_nac[['codigo_municipio', 'municipio', 'periodo', 'nacionalidad', 'poblacion']]

## 17. Ordenar el dataset

In [32]:
df_nac = df_nac.sort_values(['codigo_municipio', 'periodo'])

In [33]:
df_nac.head()

,codigo_municipio,municipio,periodo,nacionalidad,poblacion
85539,01001,Alegría-Dulantzi,2003,Española,1677.0
85619,01001,Alegría-Dulantzi,2003,Extranjera,30.0
85538,01001,Alegría-Dulantzi,2004,Española,1863.0
85618,01001,Alegría-Dulantzi,2004,Extranjera,56.0
85537,01001,Alegría-Dulantzi,2005,Española,1969.0


## 18. Exportación del dataset limpio

In [34]:
df_nac.to_csv("../data_processed/nacionalidad_limpio.csv", index=False)